# Nota del portafolio

Proyecto académico de Santiago Lara. Esta copia conserva el código original y elimina salidas guardadas y conclusiones numéricas no verificadas. No se ha vuelto a ejecutar en la preparación del portafolio porque falta el CSV original. Consultar README para datos, ejecución y limitaciones.

# **Análisis Exploratorio de Datos — Ventas de Walmart**

**Santiago Lara**

**Introducción a ciencia de datos**

**Lead University**

**Prof. Alejandro Zamora**

**12 de junio de 2026**

---

**Dataset:** Walmart Sales Dataset — [Kaggle](https://www.kaggle.com/datasets/yasserh/walmart-dataset)

---

## **1. Introducción**





El dataset utilizado es el Walmart Sales Dataset, disponible en Kaggle. Contiene registros de ventas semanales de 45 tiendas de Walmart en distintas regiones de Estados Unidos, desde febrero de 2010 hasta octubre de 2012. Además de las ventas, incluye variables del entorno como temperatura, precio del combustible, índice de precios al consumidor (CPI) y tasa de desempleo, junto con un indicador de semanas festivas.

---



Objetivo del análisis: Explorar y comprender el comportamiento de las ventas de Walmart, identificando patrones temporales, diferencias entre tiendas, el efecto de las semanas festivas y posibles relaciones con variables económicas externas.


---



¿Qué se espera encontrar?
- Estacionalidad anual con picos en noviembre y diciembre (Thanksgiving y Navidad).
- Un incremento visible en ventas durante semanas festivas.
- Alta variabilidad entre tiendas según su ubicación y tamaño.
- Baja correlación de las ventas con variables macroeconómicas en el agregado.

---



## **2. Carga de Datos**

### 2.1 Importación de librerías

Se utiliza `pandas` y `numpy` para manipulación de datos, `matplotlib` y `seaborn` para visualización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 110

### 2.2 Carga del archivo

Se carga el archivo `Walmart_Sales.csv` y se convierte la columna `Date` al tipo `datetime`.

In [ ]:
df = pd.read_csv('Walmart_Sales.csv')

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

df.head(10)

### 2.3 Dimensiones, tipos de dato y rango temporal

In [ ]:
print(f'Filas:    {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
print(f'Período:  {df["Date"].min().date()}  →  {df["Date"].max().date()}')
print(f'Tiendas:  {df["Store"].nunique()}')
print()
print('Nombre de columnas:')
print(df.columns.tolist())
print()
print('Tipos de dato:')
print(df.dtypes)


### 2.4 Valores faltantes y duplicados

In [ ]:
print('Valores nulos por columna:')
print(df.isnull().sum())
print()
print(f'Registros duplicados: {df.duplicated().sum()}')
print()
print('Distribución de Holiday_Flag:')
print(df['Holiday_Flag'].value_counts())
print(f'Porcentaje festivo: {df["Holiday_Flag"].mean()*100:.2f}%')

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

## **3. Limpieza y Revisión Inicial**

### 3.1 Estadísticas descriptivas

In [ ]:
df.describe().round(2)

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 3.2 Revisión de valores atípicos


In [ ]:
print('Percentiles de Weekly_Sales:')
print(df['Weekly_Sales'].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
      .apply(lambda x: f'${x:,.0f}'))

print()
print('Registro con mayor venta:')
idx = df['Weekly_Sales'].idxmax()
print(df.loc[idx, ['Store','Date','Weekly_Sales','Holiday_Flag']])

print()
print('Registro con menor venta:')
idx = df['Weekly_Sales'].idxmin()
print(df.loc[idx, ['Store','Date','Weekly_Sales','Holiday_Flag']])

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 3.3 Variables derivadas


In [ ]:
df['Year']         = df['Date'].dt.year
df['Month']        = df['Date'].dt.month
df['Quarter']      = df['Date'].dt.quarter
df['Month_Name']   = df['Date'].dt.strftime('%b')
df['Holiday_Label']= df['Holiday_Flag'].map({0:'No Festiva', 1:'Festiva'})

df[['Date','Year','Month','Quarter','Holiday_Label']].head()

# **4. Análisis Exploratorio de Datos**

### 4.1 Distribución de ventas semanales

Acá analizo la distribución general de ventas para entender el rango de valores y la forma de la distribución.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(df['Weekly_Sales'] / 1e3, bins=60,
        color='#1565C0', edgecolor='white', linewidth=0.3, alpha=0.85)

media   = df['Weekly_Sales'].mean() / 1e3
mediana = df['Weekly_Sales'].median() / 1e3
ax.axvline(media,   color='#E53935', linestyle='--', lw=2, label=f'Media:   ${media:,.1f}K')
ax.axvline(mediana, color='#FF8F00', linestyle='--', lw=2, label=f'Mediana: ${mediana:,.1f}K')

ax.set_xlabel('Ventas Semanales por Tienda (Miles USD)', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribución de Ventas Semanales por Tienda', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}K'))
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.2 Ventas totales por tienda

In [ ]:
from matplotlib.patches import Patch

store_sales = df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#1565C0' if i < 5 else ('#42A5F5' if i < 15 else '#90CAF9')
          for i in range(len(store_sales))]
ax.bar(store_sales.index.astype(str), store_sales.values / 1e6,
       color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Número de Tienda', fontsize=12)
ax.set_ylabel('Ventas Totales (Millones USD)', fontsize=12)
ax.set_title('Ventas Totales Acumuladas por Tienda (2010–2012)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
plt.xticks(rotation=45, fontsize=8)
ax.legend(handles=[
    Patch(facecolor='#1565C0', label='Top 5'),
    Patch(facecolor='#42A5F5', label='Top 6–15'),
    Patch(facecolor='#90CAF9', label='Resto')
], loc='upper right')
plt.tight_layout()
plt.show()

top10 = store_sales.head(10).reset_index()
top10.columns = ['Tienda','Ventas Totales']
top10['Millones']   = top10['Ventas Totales'].apply(lambda x: f'${x/1e6:.1f}M')
top10['% del total']= (top10['Ventas Totales'] / store_sales.sum() * 100).apply(lambda x: f'{x:.2f}%')
print('Top 10 tiendas:')
print(top10[['Tienda','Millones','% del total']].to_string(index=False))

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.3 Evolución temporal de las ventas

In [ ]:
weekly_total  = df.groupby('Date')['Weekly_Sales'].sum() / 1e6
holiday_weeks = df[df['Holiday_Flag'] == 1]['Date'].unique()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_total.index, weekly_total.values,
        color='#1565C0', lw=1.5, alpha=0.9, label='Ventas semanales totales')
ax.fill_between(weekly_total.index, weekly_total.values, alpha=0.12, color='#1565C0')

first = True
for hd in holiday_weeks:
    if hd in weekly_total.index:
        ax.axvline(hd, color='#E53935', alpha=0.5, lw=1.2, linestyle='--',
                   label='Semana festiva' if first else '')
        first = False

ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Ventas Totales (Millones USD)', fontsize=12)
ax.set_title('Evolución de Ventas Semanales Totales (45 tiendas)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.4 Ventas promedio por mes

In [ ]:
month_es    = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
monthly_avg = df.groupby('Month')['Weekly_Sales'].mean().reset_index()
monthly_avg['Mes'] = monthly_avg['Month'].map(month_es)

fig, ax = plt.subplots(figsize=(11, 5))
colors_m = ['#E53935' if m in [11,12] else ('#1976D2' if m in [6,7,8,9] else '#42A5F5')
            for m in monthly_avg['Month']]
bars = ax.bar(monthly_avg['Mes'], monthly_avg['Weekly_Sales'] / 1e3,
              color=colors_m, edgecolor='white', linewidth=0.5, width=0.7)
for bar, val in zip(bars, monthly_avg['Weekly_Sales'] / 1e3):
    ax.text(bar.get_x() + bar.get_width()/2, val + 8,
            f'${val:.0f}K', ha='center', fontsize=8)
ax.set_xlabel('Mes', fontsize=12)
ax.set_ylabel('Venta Promedio por Tienda (Miles USD)', fontsize=12)
ax.set_title('Ventas Promedio por Mes', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))
ax.legend(handles=[
    Patch(facecolor='#E53935', label='Temporada festiva (Nov–Dic)'),
    Patch(facecolor='#1976D2', label='Temporada media (Jun–Sep)'),
    Patch(facecolor='#42A5F5', label='Temporada baja')
], fontsize=9)
plt.tight_layout()
plt.show()

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.5 Ventas totales por año

In [ ]:
yearly = df.groupby('Year')['Weekly_Sales'].sum() / 1e6

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(yearly.index.astype(str), yearly.values,
              color=['#1565C0','#1976D2','#90CAF9'],
              edgecolor='white', linewidth=0.5, width=0.5)
for bar, val in zip(bars, yearly.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 15,
            f'${val:.1f}M', ha='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Año', fontsize=12)
ax.set_ylabel('Ventas Totales (Millones USD)', fontsize=12)
ax.set_title('Ventas Totales Anuales', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}M'))
ax.annotate('* Solo hasta octubre', xy=(2, yearly.iloc[2]),
            xytext=(1.4, yearly.iloc[2] - 280), fontsize=9, color='#888888',
            arrowprops=dict(arrowstyle='->', color='#aaa'))
plt.tight_layout()
plt.show()

crecimiento = (yearly.iloc[1] - yearly.iloc[0]) / yearly.iloc[0] * 100
print(f'Crecimiento 2010 → 2011: +{crecimiento:.1f}%')

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.6 Semanas festivas vs. no festivas

In [ ]:
stats_h = df.groupby('Holiday_Label')['Weekly_Sales'].agg(['mean','median','std','count'])
stats_h.columns = ['Media','Mediana','Desv. Est.','Registros']
print(stats_h.round(0))

mean_f  = stats_h.loc['Festiva',    'Media']
mean_nf = stats_h.loc['No Festiva', 'Media']
print(f'\nIncremento promedio en semanas festivas: +{(mean_f/mean_nf - 1)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
labels_h = ['No Festiva', 'Festiva']
colors_h = ['#1565C0', '#E53935']

vals_mean = [stats_h.loc['No Festiva','Media']/1e3,  stats_h.loc['Festiva','Media']/1e3]
vals_med  = [stats_h.loc['No Festiva','Mediana']/1e3, stats_h.loc['Festiva','Mediana']/1e3]

for ax, vals, titulo in zip(axes, [vals_mean, vals_med], ['Venta Promedio', 'Venta Mediana']):
    bars = ax.bar(labels_h, vals, color=colors_h, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 8,
                f'${v:,.1f}K', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Miles USD', fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

plt.suptitle('Semanas Festivas vs. No Festivas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.7 Mapa de calor de correlaciones

In [ ]:
corr_cols  = ['Weekly_Sales','Temperature','Fuel_Price','CPI','Unemployment','Holiday_Flag']
labels_map = {'Weekly_Sales':'Ventas\nSemanales','Temperature':'Temperatura',
              'Fuel_Price':'Precio\nCombustible','CPI':'IPC',
              'Unemployment':'Desempleo','Holiday_Flag':'Semana\nFestiva'}

corr_m = df[corr_cols].corr()
corr_m.index   = [labels_map[c] for c in corr_m.index]
corr_m.columns = [labels_map[c] for c in corr_m.columns]

mask = np.triu(np.ones_like(corr_m, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_m, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            ax=ax, linewidths=0.5, linecolor='white', vmin=-1, vmax=1,
            annot_kws={'size': 11})
ax.set_title('Correlaciones entre Variables (Pearson)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('Correlación con Ventas Semanales:')
corr_v = df[corr_cols].corr()['Weekly_Sales'].drop('Weekly_Sales').sort_values(ascending=False)
for var, val in corr_v.items():
    print(f'  {var:<20} r = {val:+.4f}')

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.8 Diagramas de dispersión: ventas vs variables del entorno

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
variables = ['Temperature','Fuel_Price','CPI','Unemployment']
etiquetas = ['Temperatura (°F)','Precio Combustible (USD/gal)','IPC','Desempleo (%)']
sample = df.sample(n=1500, random_state=42)

for ax, var, label in zip(axes.flat, variables, etiquetas):
    c_pts = sample['Holiday_Flag'].map({0:'#1565C0', 1:'#E53935'})
    ax.scatter(sample[var], sample['Weekly_Sales']/1e3,
               c=c_pts, alpha=0.35, s=15, edgecolors='none')
    z = np.polyfit(sample[var], sample['Weekly_Sales']/1e3, 1)
    xline = np.linspace(sample[var].min(), sample[var].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), color='#FF8F00', lw=2, linestyle='--')
    r = df[var].corr(df['Weekly_Sales'])
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Ventas (Miles USD)', fontsize=10)
    ax.set_title(f'Ventas vs {label.split(" (")[0]}  (r = {r:.3f})', fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))

fig.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#1565C0', ms=8, label='No Festiva'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#E53935',  ms=8, label='Festiva'),
    Line2D([0],[0], color='#FF8F00', lw=2, linestyle='--', label='Tendencia')
], loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.04))
plt.suptitle('Ventas vs Variables del Entorno', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

### 4.9 Distribución de ventas por trimestre

In [ ]:
q_labels = {1:'Q1 (Ene–Mar)', 2:'Q2 (Abr–Jun)', 3:'Q3 (Jul–Sep)', 4:'Q4 (Oct–Dic)'}
df['Quarter_Label'] = df['Quarter'].map(q_labels)
order = list(q_labels.values())

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Quarter_Label', y='Weekly_Sales', order=order,
            palette=['#90CAF9','#42A5F5','#1976D2','#E53935'], ax=ax,
            width=0.5, flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_xlabel('Trimestre', fontsize=12)
ax.set_ylabel('Ventas Semanales por Tienda (USD)', fontsize=12)
ax.set_title('Distribución de Ventas por Trimestre', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

q_stats = df.groupby('Quarter')['Weekly_Sales'].agg(['mean','median']).round(0)
q_stats.index = order
q_stats.columns = ['Media','Mediana']
print(q_stats.applymap(lambda x: f'${x:,.0f}').to_string())

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

## **5. Resultados y Hallazgos**

Los resultados de esta sección deben interpretarse después de ejecutar el cuaderno con el dataset original. Las comparaciones describen asociaciones y no demuestran causalidad.

## **6. Conclusiones**

**Conclusión 1: Las ventas presentan estacionalidad anual consistente**  
Diciembre y noviembre registran sistemáticamente los promedios más altos (~35% por encima del promedio anual) y enero y febrero los más bajos. Este patrón se repite en los tres años del dataset.

**Conclusión 2: Las semanas festivas generan mayor volumen de ventas**  
Las semanas festivas producen en promedio un +20.2% más en ventas que las semanas ordinarias, efecto consistente tanto en media como en mediana a lo largo de los tres años analizados.

**Conclusión 3: Existe alta variabilidad en ventas entre tiendas**  
La tienda de mayor rendimiento acumula casi 4 veces más ventas que la de menor rendimiento. Las 10 tiendas más grandes concentran el 35% de las ventas totales de la cadena.

**Conclusión 4: Las variables macroeconómicas no muestran correlación significativa con las ventas**  
Temperatura, precio del combustible, IPC y desempleo presentan correlaciones cercanas a cero con las ventas semanales. La única variable con correlación observable es `Holiday_Flag` (r = +0.11).

**Conclusión 5: El dataset no presenta problemas de calidad**  
No se encontraron valores nulos ni registros duplicados. Los valores extremos de ventas corresponden a semanas festivas y no representan errores de datos.